# 富邦期貨日內行情驗證
STATIC TESTED / COLAB/API NOT YET TESTED（開發交付狀態）。
設定左側 Secrets 的四個 FUBON 名稱並允許此 Notebook 存取。憑證先放私人 Drive。
Run All：僅行情讀取，無交易；首次須網路安裝官方 SDK。正式執行必須 clean Git。
本 Notebook 支援 Python 3.8–3.13；研究測試以 3.11 為基準，Colab 不支援版本會明確停止。
所有 raw/diagnostics 從取得 response 起直接寫入獨立 Drive 目錄，中斷後仍可保留。


## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone / identify repository

In [ ]:
from pathlib import Path
import os, subprocess, sys, tempfile
from google.colab import userdata
REPO_URL = "https://github.com/hh4832/fubon_futures_intraday_data_validation.git"
REPO_DIR = Path('/content/fubon_futures_intraday_data_validation')
# Public clone first; only a private repo needs optional GITHUB_TOKEN.
if not REPO_DIR.exists():
    result = subprocess.run(['git','clone',REPO_URL,str(REPO_DIR)],capture_output=True,text=True)
    if result.returncode:
        try:
            token = userdata.get('GITHUB_TOKEN')
        except Exception:
            raise RuntimeError('Public clone failed; private repositories need GITHUB_TOKEN') from None
        if not token:
            raise RuntimeError('GITHUB_TOKEN unavailable')
        with tempfile.TemporaryDirectory() as tmp:
            askpass = Path(tmp)/'askpass.py'
            askpass.write_text('#!/usr/bin/env python3\nimport os,sys\nprint("x-access-token" if "Username" in sys.argv[1] else os.environ["COLAB_CLONE_TOKEN"])\n')
            askpass.chmod(0o700)
            env = dict(os.environ,GIT_ASKPASS=str(askpass),GIT_TERMINAL_PROMPT='0',COLAB_CLONE_TOKEN=token)
            retry = subprocess.run(['git','clone',REPO_URL,str(REPO_DIR)],env=env,capture_output=True)
            del env,token
            if retry.returncode:
                raise RuntimeError('Authenticated clone failed; output suppressed')
remote = subprocess.check_output(['git','remote','get-url','origin'],cwd=REPO_DIR,text=True).strip()
if remote != REPO_URL:
    raise RuntimeError('Repository origin differs from expected URL')
os.chdir(REPO_DIR)
sys.path.insert(0,str(REPO_DIR))
subprocess.run([sys.executable,'-m','pip','install','-r','requirements.txt'],check=True)


## 3. Git provenance

In [ ]:
from src.archive import git_info
provenance = git_info(REPO_DIR)
print(provenance)
if provenance['git_status']:
    raise RuntimeError('Use a clean Git checkout; no auto pull or overwrite')

## 4. Environment diagnostics

In [ ]:
import platform
print({'python':sys.version,'executable':sys.executable,'platform':platform.platform(),'architecture':platform.machine()})
from src.config import load_config
cfg = load_config()

## 5. Official SDK installation

In [ ]:
from tqdm.auto import tqdm
from scripts.install_fubon_sdk_colab import install
for _ in tqdm(range(1),desc='SDK setup'):
    sdk_info = install(cfg['sdk_version'],cfg['sdk_download_url'],cfg['sdk_official_sha256'])

## 6. Import smoke test

In [ ]:
import fubon_neo
from fubon_neo.sdk import FubonSDK
print("SDK import PASS")

## 7. Colab Secrets

In [ ]:
from src.fubon_auth import load_secrets, validate_certificate, login
secrets = load_secrets(userdata.get)

## 8. Certificate path

In [ ]:
validate_certificate(secrets)

## 9. API Key login

In [ ]:
sdk, auth_info = login(secrets)
del secrets

## 10. Market-data initialization / new archive

In [ ]:
try:
    sdk.init_realtime()
    restfutopt = sdk.marketdata.rest_client.futopt
except Exception:
    raise RuntimeError('Market-data initialization failed; SDK exception suppressed') from None
from src.archive import start_run, finish_run
from src.fubon_client import MarketReader
run_dir, run_info = start_run(REPO_DIR,cfg,sdk_info,auth_info)
reader = MarketReader(restfutopt,run_dir,cfg)
print('Output:',run_dir)


## 11. Contract discovery

In [ ]:
from src.discover_contracts import discover, select_contract
from src.diagnostics import write_table
contract_responses, contracts = discover(reader)
contracts.to_csv(run_dir/'diagnostics'/'discovered_contracts.csv',index=False) if not contracts.empty else write_table(run_dir,'discovered_contracts',[],['symbol','status'])
finish_run(reader,run_info,status='IN_PROGRESS')


## 12. Inspect raw discovery

In [ ]:
for name, response in contract_responses.items():
    print(name)
    print(type(response))
    print(response)
    if isinstance(response,dict):
        print(response.keys())
display(contracts)

## 13. Select discovered TXF contract

In [ ]:
try:
    symbol, reason = select_contract(contracts,cfg['selected_symbol'])
except Exception:
    from src.archive import write_info
    run_info['execution_status'] = 'DISCOVERY_UNRESOLVED'
    finish_run(reader,run_info,status='DISCOVERY_UNRESOLVED')
    raise
run_info.update(contract_symbol=symbol,selection_reason=reason)
print(symbol,reason)


## 14. Fetch eight intraday combinations

In [ ]:
from src.fetch_intraday import fetch
items = fetch(reader,symbol,cfg)
finish_run(reader,run_info,status='IN_PROGRESS')

## 15. Raw response inspection

In [ ]:
for item in items:
    response = item["response"]
    print(item["symbol"],item["session"],item["timeframe"])
    print(type(response))
    print(response)
    if isinstance(response,dict):
        print(response.keys())

## 16. Normalize

In [ ]:
from src.pipeline import Validation
validation = Validation(items,run_dir,cfg)
display(validation.stage('normalization'))

## 17. schema validation

In [ ]:
display(validation.stage('schema validation'))

## 18. OHLC validation

In [ ]:
display(validation.stage('OHLC validation'))

## 19. timestamp validation

In [ ]:
display(validation.stage('timestamp validation'))

## 20. session validation

In [ ]:
display(validation.stage('session validation'))

## 21. Aggregation alignment / comparison

In [ ]:
display(validation.aggregation())

## 22. Historical capability (one unsupported date probe)

In [ ]:
from src.diagnostics import historical_capability
history = historical_capability(reader,symbol,cfg)
print(history)

## 23. Summary

In [ ]:
from src.diagnostics import write_summary
summary = write_summary(run_dir,run_info,validation.tables,history)
print(summary)

## 24. Verify Drive archive

In [ ]:
for _ in tqdm(range(1),desc='archive outputs'):
    finish_run(reader,run_info)

## 25. Final output directory

In [ ]:
print(str(run_dir.resolve()))
print('Files:',*[str(p.relative_to(run_dir)) for p in sorted(run_dir.rglob('*')) if p.is_file()],sep='\n')